In [209]:
from statsmodels.stats.multitest import multipletests
from scipy.stats import mannwhitneyu, kruskal
from collections import defaultdict
from datetime import datetime
import pandas as pd
import numpy as np
import re
import json
import os


In [210]:
data_dir = "./data"
processed_dir = os.path.join(data_dir, "processed")
session_dir = os.path.join(processed_dir, "merged")

gameplay_dir = os.path.join(session_dir, "Gameplay")
traces_path = os.path.join(gameplay_dir, "traces.ndjson")


In [211]:
with open(traces_path, "r", encoding="utf-8") as f:
	print(repr(f.read()[:200]))
	

'{"actor": {"account": {"name": "684837aae48b5a00221a37c9_fwme", "homePage": "https://simva-beta.e-ucm.es"}}, "result": {"extensions": {"https://w3id.org/xapi/seriousgame/extensions/Sexuality": "hetero'


In [212]:
traces = []
invalid_count = 0

with open(traces_path, "r", encoding="utf-8") as f:
	for line in f:
		line = line.strip()
		if line:
			try:
				traces.append(json.loads(line))
			except json.JSONDecodeError:
				invalid_count += 1

print("Total valid traces:", len(traces))
print("Total invalid traces:", invalid_count)


Total valid traces: 98930
Total invalid traces: 0


In [213]:
NODE_EXT_ID = "https://w3id.org/xapi/seriousgame/extensions/Node"
RESPONSE_EXT_ID = "https://w3id.org/xapi/seriousgame/extensions/Response"
CHOICE_OBJ_ID = "https://w3id.org/xapi/seriousgames/activity-types/dialog-tree/OptionSelect"


In [214]:
target_nodes = {
	"Scene6Bedroom.phone.choices2": "send_more_nudes",
	"Scene6BedroomRouteA1.phone.choices2": "agree_to_meet_harasser",
	"Scene6LunchRouteB.interruption.choices": "confess_to_parents"
}

END_NODES = {
	"END_MEET": "meet_harasser_in_person",
	"END_TELL": "confess_truth",
	"END_LIE": "hide_truth"
}


def parse_time(timestamp):
	if not timestamp:
		return datetime.fromisoformat("1970-01-01T00:00:00+00:00")

	return datetime.fromisoformat(timestamp.replace("Z", "+00:00"))


def classify_path(path):
	options = [option for _, option in path]

	if "Enviar otra foto." in options and "Aceptar." in options:
		return "END_MEET"

	if "Decir la verdad." in options:
		return "END_TELL"

	if "Mentir." in options:
		return "END_LIE"

	return None


def extract_sequence(traces):
	traces = sorted(
		traces,
		key=lambda trace: parse_time(trace.get("timestamp"))
	)

	sequence = []

	for trace in traces:
		if trace.get("object", {}).get("id") != CHOICE_OBJ_ID:
			continue

		extensions = trace.get("result", {}).get("extensions", {}) or {}

		node = extensions.get(NODE_EXT_ID)
		option = extensions.get(RESPONSE_EXT_ID)

		if node in target_nodes and option:
			sequence.append((node, option))

	return sequence


# Agrupar trazas por usuario
user_traces = defaultdict(list)

for trace in traces:
	user = trace.get("actor", {}).get("account", {}).get("name")

	if user:
		user_traces[user].append(trace)

# Usuarios de cada final
users_by_end = defaultdict(list)

for user, user_trace_list in user_traces.items():
	path = extract_sequence(user_trace_list)
	end = classify_path(path)

	if end:
		users_by_end[end].append(user)


# Mostrar resultados
for end, users in users_by_end.items(): 
	print(f"{END_NODES[end]}: {len(users)} usuarios")
		

meet_harasser_in_person: 33 usuarios
hide_truth: 9 usuarios
confess_truth: 62 usuarios


In [215]:
def load_json(path):
	with open(path, encoding="utf-8") as f:
		return json.load(f)

def load_tests(base_dir: str):
	data = defaultdict(lambda: {
		"pre": {"code": None, "full": None},
		"post": {"code": None, "full": None},
	})

	for phase in ["Pre", "Post"]:
		phase_key = phase.lower()
		phase_dir = os.path.join(base_dir, phase)

		for file in os.listdir(phase_dir):
			if file.endswith(".json"):
				file_path = os.path.join(phase_dir, file)
				content = load_json(file_path)

				kind = "code" if "code" in file else "full"

				for user_id, user_data in content.items():
					data[user_id][phase_key][kind] = user_data

	return dict(data)


In [244]:
def parse_likert(v: str):
	m = re.match(r"AO0?(\d)", v)
	if m:
		return int(m.group(1))

	return None

def extract_data(code: dict, full: dict):
	out = {}
	items = {}

	age = pd.to_numeric(full.get("Edad"), errors="coerce")

	if pd.isna(age):
		age = None
	else:
		age = int(age)

	course = full.get("Curso")
	school_year = 2

	if course:
		match = re.search(r"\d+", course)
		if match:
			school_year = int(match.group())
			
	full_keys = list(full.keys())

	for idx, (k, v) in enumerate(code.items()):
		if k.startswith("G02Q03[SQ"):		
			m = re.search(r"\[SQ(\d+)\]", k)
			if m:
				q_id = f"Q{m.group(1)}"
				out[q_id] = parse_likert(v)
				items[q_id] = full_keys[idx]

	return items, out, age, school_year

In [245]:
tests = load_tests(session_dir)


In [246]:
rows = []

for user_id, test in tests.items():
	pre_code = test["pre"]["code"]
	pre_full = test["pre"]["full"]
	post_code = test["post"]["code"]
	post_full = test["post"]["full"]

	if not all([pre_code, pre_full, post_code, post_full]):
		print("Skipping:", user_id)
		continue

	pre_items, pre_vals, age, school_year = extract_data(pre_code, pre_full)
	post_items, post_vals, _, _ = extract_data(post_code, post_full)

	common_ids = set(pre_vals) & set(post_vals)

	for id in common_ids:
		rows.append({
			"user_id": user_id,
			"age": age,
			"school_year": school_year,
			"id": id,
			"item": pre_items[id],
			"pre": pre_vals[id],
			"post": post_vals[id]
		})

global_df = pd.DataFrame(rows)
global_df["diff"] = global_df["post"] - global_df["pre"]

print("Total users:", global_df["user_id"].nunique())
print("Total rows:", len(global_df))

display(global_df.head())


Total users: 104
Total rows: 2080


,user_id,age,school_year,id,item,pre,post,diff
0,684837aae48b5a00221a37c9_uzdq,13,2,Q11,¿Cómo de peligrosas consideras las siguientes ...,5,4,-1
1,684837aae48b5a00221a37c9_uzdq,13,2,Q18,¿Cómo de peligrosas consideras las siguientes ...,2,2,0
2,684837aae48b5a00221a37c9_uzdq,13,2,Q09,¿Cómo de peligrosas consideras las siguientes ...,4,4,0
3,684837aae48b5a00221a37c9_uzdq,13,2,Q06,¿Cómo de peligrosas consideras las siguientes ...,2,4,2
4,684837aae48b5a00221a37c9_uzdq,13,2,Q14,¿Cómo de peligrosas consideras las siguientes ...,2,3,1


In [247]:
for _, row in global_df.drop_duplicates(subset="id").sort_values("id").iterrows():
    print(row["id"], row["item"])
	

Q01 ¿Cómo de peligrosas consideras las siguientes acciones? [Tener tu perfil en público]
Q02 ¿Cómo de peligrosas consideras las siguientes acciones? [Agregar a contactos o aceptar como seguidores a personas que no conoces en la vida real]
Q03 ¿Cómo de peligrosas consideras las siguientes acciones? [Unirte a grupos o comunidades o servidores públicos]
Q04 ¿Cómo de peligrosas consideras las siguientes acciones? [Hablar por chat (escrito o de voz) con alguien que no conoces en la vida real]
Q05 ¿Cómo de peligrosas consideras las siguientes acciones? [Hablar por videollamada con alguien que no conoces en la vida real]
Q06 ¿Cómo de peligrosas consideras las siguientes acciones? [Entablar amistad con alguien que no conoces en la vida real]
Q07 ¿Cómo de peligrosas consideras las siguientes acciones? [Quedar en la vida real con una persona que has conocido en internet]
Q08 ¿Cómo de peligrosas consideras las siguientes acciones? [Hablar con adultos por internet]
Q09 ¿Cómo de peligrosas consider

In [248]:
user_to_group = {}

for user in users_by_end["END_MEET"]:
	user_to_group[user] = "MEET"

for user in users_by_end["END_TELL"]:
	user_to_group[user] = "TELL"

for user in users_by_end["END_LIE"]:
	user_to_group[user] = "LIE"

df = global_df.copy()

df["group"] = df["user_id"].map(user_to_group)

# Kruskal-Wallis para cada ítem
results = []
for item_id, df_item in df.groupby("id"):

	groups = {
		group: data["diff"].to_numpy()
		for group, data in df_item.groupby("group")
	}

	# Necesitamos los 3 grupos
	if not all(group in groups for group in ["MEET", "TELL", "LIE"]):
		continue

	# Al menos 2 participantes por grupo
	if any(len(values) < 2 for values in groups.values()):
		continue

	# Test de Kruskal-Wallis
	# Se utiliza para comparar una variable entre tres o más grupos independientemente de la normalidad.
	# En este análisis se utiliza para comprobar si el cambio pre-post difiere entre los tres finales: MEET, TELL y LIE.
	# Hipótesis nula (H0): la distribución del cambio pre-post es la misma en los tres grupos.
	# Hipótesis alternativa (H1): al menos uno de los tres grupos presenta una distribución del cambio pre-post diferente.
	# Si p < 0.05, se rechaza H0 y existe evidencia de que al menos uno de los grupos difiere de los demás.
	# El test de Kruskal-Wallis únicamente indica si existe una diferencia global entre los tres grupos, 
	# pero no identifica qué grupos difieren entre sí. 
	# Por ello, cuando el resultado es significativo, se realizan comparaciones por pares mediante el test de Mann-Whitney U.
	stat, p = kruskal(
		groups["MEET"],
		groups["TELL"],
		groups["LIE"]
	)

	results.append({
		"id": item_id,
		
		"n_MEET": len(groups["MEET"]),
		"n_TELL": len(groups["TELL"]),
		"n_LIE": len(groups["LIE"]),

		"mean_diff_MEET": np.mean(groups["MEET"]),
		"mean_diff_TELL": np.mean(groups["TELL"]),
		"mean_diff_LIE": np.mean(groups["LIE"]),

		"median_diff_MEET": np.median(groups["MEET"]),
		"median_diff_TELL": np.median(groups["TELL"]),
		"median_diff_LIE": np.median(groups["LIE"]),

		"stat": stat,
		"p": p
	})


comparison_df = pd.DataFrame(results)

# https://cienciadedatos.net/documentos/19b_comparaciones_multiples_correccion_p-value_fdr
# FDR (Benjamini-Hochberg) para los 20 tests de Kruskal-Wallis.
# Se realiza un test de Kruskal-Wallis independiente para cada ítem Likert.
# Al realizar múltiples tests, aumenta la probabilidad de obtener falsos positivos por el azar.
# En cada test, si se fija alpha = 0.05, existe un 5% de probabilidad
# de rechazar la hipótesis nula cuando es verdadera.
# Al realizar 20 tests, esta probabilidad acumulada de obtener falsos positivos aumenta.
# La corrección de Benjamini-Hochberg ajusta la False Discovery Rate (FDR)
comparison_df["p_adj"] = multipletests(comparison_df["p"], method="fdr_bh")[1]

comparison_df["significant"] = comparison_df["p_adj"] < 0.05

# Seleccionar ítems para los post-hoc
# Se hacen post-hoc solamente cuando el Kruskal-Wallis tiene p_adj < 0.05
significant_items = comparison_df.loc[
	comparison_df["p_adj"] < 0.05,
	"id"
].tolist()

# Post-hoc por pares
pairwise_results = []

pairs = [
	("MEET", "TELL"),
	("MEET", "LIE"),
	("TELL", "LIE")
]

for item_id in significant_items:
	df_item = df[df["id"] == item_id]
	
	groups = {
		group: data["diff"].to_numpy()
		for group, data in df_item.groupby("group")
	}

	for group1, group2 in pairs:
		x = groups[group1]
		y = groups[group2]

		if len(x) < 2 or len(y) < 2:
			continue

		# Test de Mann-Whitney U
		# Se utiliza para comparar una variable no parametríca entre dos grupos independientes.
		# Hipótesis nula (H0): la distribución del cambio pre-post es la misma en los dos grupos comparados.
		# Hipótesis alternativa (H1): la distribución del cambio pre-post es diferente entre los dos grupos.
		stat, p = mannwhitneyu(
			x,
			y,
			alternative="two-sided"
		)

		pairwise_results.append({
			"id": item_id,
			"group1": group1,
			"group2": group2,
			"n1": len(x),
			"n2": len(y),
			"median1": np.median(x),
			"median2": np.median(y),
			"stat": stat,
			"p": p
		})


pairwise_df = pd.DataFrame(pairwise_results)

# Cada pregunta constituye una familia de hipótesis independiente.
# Por este motivo, la corrección FDR se aplica por separado dentro
# de cada pregunta, considerando los 3 conjuntos:
# - MEET vs TELL
# - MEET vs LIE
# - TELL vs LIE
if not pairwise_df.empty:
	pairwise_df["p_adj"] = np.nan

	for item_id, indices in pairwise_df.groupby("id").groups.items():
		p_values = pairwise_df.loc[indices, "p"]

		# FDR de los post-hoc
		pairwise_df.loc[indices, "p_adj"] = multipletests(p_values, method="fdr_bh")[1]

	pairwise_df["significant"] = pairwise_df["p_adj"] < 0.05

# Resultados
print("COMPARACIÓN GLOBAL: KRUSKAL-WALLIS")
display(comparison_df)

print("POST-HOC MANN-WHITNEY")
display(pairwise_df)


COMPARACIÓN GLOBAL: KRUSKAL-WALLIS


,id,n_MEET,n_TELL,n_LIE,mean_diff_MEET,mean_diff_TELL,mean_diff_LIE,median_diff_MEET,median_diff_TELL,median_diff_LIE,stat,p,p_adj,significant
0,Q01,33,62,9,0.424242,0.693548,1.000000,0.0,1.0,1.0,2.379730,0.304262,0.547303,False
1,Q02,33,62,9,0.515152,0.338710,0.555556,0.0,0.0,0.0,2.227156,0.328382,0.547303,False
2,Q03,33,62,9,0.666667,0.758065,1.333333,1.0,1.0,1.0,1.119264,0.571419,0.761892,False
3,Q04,33,62,9,0.393939,0.548387,-0.111111,0.0,0.0,0.0,2.834498,0.242380,0.547303,False
4,Q05,33,62,9,0.212121,0.177419,0.111111,0.0,0.0,0.0,0.265138,0.875842,0.879191,False
5,Q06,33,62,9,0.696970,0.693548,0.222222,1.0,1.0,0.0,2.241939,0.325964,0.547303,False
6,Q07,33,62,9,0.515152,0.096774,0.222222,0.0,0.0,0.0,3.064238,0.216077,0.547303,False
7,Q08,33,62,9,0.454545,0.225806,0.222222,0.0,0.0,0.0,2.347117,0.309264,0.547303,False
8,Q09,33,62,9,0.606061,0.725806,0.888889,0.0,1.0,0.0,1.237194,0.538700,0.761892,False
9,Q10,33,62,9,0.757576,0.741935,1.111111,1.0,1.0,1.0,0.400856,0.818381,0.879191,False


POST-HOC MANN-WHITNEY


""


In [249]:
target_nodes = {
	"Scene1Bedroom1.computer2.choices2": "school_year",
	"Scene1Lunch1.main.choices": "talk_parents_first_day_school",
	"Scene4Bedroom.phone.choices": "harassment_conflict_friend",
	"Scene4Garage.photo.choices": "suspicious_gift_photo",
	"Scene4Garage.interruption.choices": "stop_sending_gift_photo",
	"Scene5Livingroom.choices": "argument_with_harasser",
	"Scene6Bedroom.phone.choices2": "send_more_nudes",
	"Scene6BedroomRouteA1.phone.choices2": "agree_to_meet_harasser",
	"Scene6LunchRouteB.interruption.choices": "confess_to_parents"
}


In [250]:
node_response_users = defaultdict(lambda: defaultdict(set))

for trace in traces:
	obj_id = trace.get("object", {}).get("id", "")

	if obj_id == CHOICE_OBJ_ID:
		extensions = trace.get("result", {}).get("extensions", {})

		node = extensions.get(NODE_EXT_ID)
		response = extensions.get(RESPONSE_EXT_ID)

		user = trace.get("actor", {}).get("account", {}).get("name")

		if node in target_nodes and response and user:
			node_response_users[node][response].add(user)
			

In [251]:
for node, responses in node_response_users.items():
	print(f"\nNodo: {node}")

	for response, users in responses.items():
		print(f"	Elección: {response}")
		print(f"	Número de usuarios: {len(users)}")
		


Nodo: Scene1Lunch1.main.choices
	Elección: Contarles sobre mi día.
	Número de usuarios: 99
	Elección: No contarles sobre mi día.
	Número de usuarios: 6

Nodo: Scene1Bedroom1.computer2.choices2
	Elección: 2º de la ESO
	Número de usuarios: 54
	Elección: 4º de la ESO
	Número de usuarios: 37
	Elección: 3º de la ESO
	Número de usuarios: 9
	Elección: 1º de la ESO
	Número de usuarios: 8

Nodo: Scene4Garage.photo.choices
	Elección: Decir la verdad.
	Número de usuarios: 88
	Elección: Mentir.
	Número de usuarios: 16

Nodo: Scene4Garage.interruption.choices
	Elección: Nadie lo va a saber.
	Número de usuarios: 86
	Elección: No te incumbe.
	Número de usuarios: 19

Nodo: Scene4Bedroom.phone.choices
	Elección: Por la foto.
	Número de usuarios: 53
	Elección: Prefiero no hablar de ello.
	Número de usuarios: 33
	Elección: Por nuestra relación.
	Número de usuarios: 20

Nodo: Scene5Livingroom.choices
	Elección: Preguntar por su reacción.
	Número de usuarios: 91
	Elección: Ofenderse.
	Número de usuarios: 

In [252]:
node_rows = []

for node, responses in node_response_users.items():
	for response, users in responses.items():
		for user in users:
			node_rows.append({
				"user_id": user,
				"node": node,
				"response": response
			})

node_df = pd.DataFrame(node_rows)

display(node_df.head())

,user_id,node,response
0,684837b0e48b5a00221a37d0_wulv,Scene1Lunch1.main.choices,Contarles sobre mi día.
1,682b4a41c76d2e0023ed4b24_xwbd,Scene1Lunch1.main.choices,Contarles sobre mi día.
2,682b4a41c76d2e0023ed4b24_lqyg,Scene1Lunch1.main.choices,Contarles sobre mi día.
3,684837b0e48b5a00221a37d0_sffq,Scene1Lunch1.main.choices,Contarles sobre mi día.
4,682b4a72c76d2e0023ed4d05_nuan,Scene1Lunch1.main.choices,Contarles sobre mi día.


In [253]:
diff_df = global_df[["user_id", "id", "diff"]].copy()

analysis_df = node_df.merge(
	diff_df,
	on="user_id",
	how="inner"
)

node_dfs = {
	node: data.copy()
	for node, data in analysis_df.groupby("node")
}


In [254]:
node_results = {}

for node, df_node in node_dfs.items():
	results = []

	responses = df_node["response"].unique()

	print(f"\n{'='*70}")
	print(f"Node: {node}")
	print(f"Responses: {list(responses)}")

	for item_id, df_item in df_node.groupby("id"):

		groups = {
			response: data["diff"].to_numpy()
			for response, data in df_item.groupby("response")
		}

		groups = {
			response: values
			for response, values in groups.items()
			if len(values) >= 2
		}

		if len(groups) >= 2:
			if len(groups) == 2:
				response_1, response_2 = groups.keys()

				# https://numiqo.es/tutorial/mann-whitney-u-test
				# Test no paramétrico que se utiliza para comprobar si existe una diferencia entre dos grupos independientes
				# La prueba convierte los valores a rangos y comprueba si las observaciones de un grupo tieneden a ser mayores o menores que las del otro
				# H0: las distribuciones de ambos grupos son iguales
				# H1: las distribuciones difieren
				stat, p = mannwhitneyu(
					groups[response_1],
					groups[response_2],
					alternative="two-sided"
				)

				result = {
					"id": item_id,
					"test": "Mann-Whitney U",
					"stat": stat,
					"p": p,
				}

			else:
				stat, p = kruskal(*groups.values())

				# Test no paramétrico que se utiliza para comparar tres o más grupos independientes
				# H0: los grupos proceden de la misma distribución
				# H1: al menos uno de los grupos difiere
				result = {
					"id": item_id,
					"test": "Kruskal-Wallis",
					"stat": stat,
					"p": p,
				}

			for response, values in groups.items():
				result[f"n_{response}"] = len(values)
				result[f"mean_diff_{response}"] = np.mean(values)
				result[f"median_diff_{response}"] = np.median(values)

			results.append(result)

	results_df = pd.DataFrame(results)

	if not results_df.empty:
		results_df["p_adj"] = multipletests(results_df["p"], method="fdr_bh")[1]

		results_df["significant"] = results_df["p_adj"] < 0.05
		
	node_results[node] = results_df
	


Node: Scene1Bedroom1.computer2.choices2
Responses: ['2º de la ESO', '4º de la ESO', '3º de la ESO', '1º de la ESO']

Node: Scene1Lunch1.main.choices
Responses: ['Contarles sobre mi día.', 'No contarles sobre mi día.']

Node: Scene4Bedroom.phone.choices
Responses: ['Por la foto.', 'Prefiero no hablar de ello.', 'Por nuestra relación.']

Node: Scene4Garage.interruption.choices
Responses: ['Nadie lo va a saber.', 'No te incumbe.']

Node: Scene4Garage.photo.choices
Responses: ['Decir la verdad.', 'Mentir.']

Node: Scene5Livingroom.choices
Responses: ['Preguntar por su reacción.', 'Ofenderse.']

Node: Scene6Bedroom.phone.choices2
Responses: ['Enviar otra foto.', 'Ya no quiero enviar más fotos.']

Node: Scene6BedroomRouteA1.phone.choices2
Responses: ['Aceptar.', 'Negarse.']

Node: Scene6LunchRouteB.interruption.choices
Responses: ['Decir la verdad.', 'Mentir.']


In [255]:
for node, df in node_results.items():
    node_name = target_nodes.get(node, node)
    options = node_dfs[node]["response"].value_counts()

    print(f"\nNode: {node_name}")
    print(f"Id: {node}")
    print(f"Options:\n{options}")
    print(f"Test: {df['test'].iloc[0]}")
    print(f"Significant: {df['significant'].sum()}/{len(df)}")

    display(df)



Node: school_year
Id: Scene1Bedroom1.computer2.choices2
Options:
response
2º de la ESO    1080
4º de la ESO     740
3º de la ESO     180
1º de la ESO     160
Name: count, dtype: int64
Test: Kruskal-Wallis
Significant: 0/20


,id,test,stat,p,n_1º de la ESO,mean_diff_1º de la ESO,median_diff_1º de la ESO,n_2º de la ESO,mean_diff_2º de la ESO,median_diff_2º de la ESO,n_3º de la ESO,mean_diff_3º de la ESO,median_diff_3º de la ESO,n_4º de la ESO,mean_diff_4º de la ESO,median_diff_4º de la ESO,p_adj,significant
0,Q01,Kruskal-Wallis,5.142778,0.161636,8,0.250,0.0,54,0.759259,0.5,9,1.000000,1.0,37,0.405405,0.0,0.637633,False
1,Q02,Kruskal-Wallis,3.435434,0.329233,8,0.250,0.0,54,0.500000,0.0,9,0.555556,1.0,37,0.243243,0.0,0.637633,False
2,Q03,Kruskal-Wallis,2.914086,0.405062,8,0.625,1.0,54,0.814815,1.0,9,1.333333,1.0,37,0.594595,0.0,0.637633,False
3,Q04,Kruskal-Wallis,5.772580,0.123214,8,-0.375,0.0,54,0.537037,0.0,9,0.666667,1.0,37,0.405405,1.0,0.637633,False
4,Q05,Kruskal-Wallis,9.308006,0.025464,8,-0.625,-0.5,54,0.203704,0.0,9,0.555556,0.0,37,0.189189,0.0,0.509281,False
5,Q06,Kruskal-Wallis,4.443966,0.217343,8,0.875,1.0,54,0.740741,1.0,9,1.000000,1.0,37,0.297297,0.0,0.637633,False
6,Q07,Kruskal-Wallis,1.715264,0.633546,8,0.000,0.0,54,0.203704,0.0,9,0.555556,0.0,37,0.189189,0.0,0.718328,False
7,Q08,Kruskal-Wallis,1.006772,0.799613,8,0.250,0.0,54,0.333333,0.0,9,0.444444,0.0,37,0.162162,0.0,0.799613,False
8,Q09,Kruskal-Wallis,1.657179,0.646495,8,0.375,0.0,54,0.796296,1.0,9,0.666667,1.0,37,0.567568,0.0,0.718328,False
9,Q10,Kruskal-Wallis,2.781613,0.426536,8,0.500,0.5,54,0.944444,1.0,9,0.555556,0.0,37,0.567568,0.0,0.637633,False



Node: talk_parents_first_day_school
Id: Scene1Lunch1.main.choices
Options:
response
Contarles sobre mi día.       1980
No contarles sobre mi día.     120
Name: count, dtype: int64
Test: Mann-Whitney U
Significant: 0/20


,id,test,stat,p,n_Contarles sobre mi día.,mean_diff_Contarles sobre mi día.,median_diff_Contarles sobre mi día.,n_No contarles sobre mi día.,mean_diff_No contarles sobre mi día.,median_diff_No contarles sobre mi día.,p_adj,significant
0,Q01,Mann-Whitney U,232.0,0.342366,99,0.626263,0.0,6,0.833333,1.0,0.902823,False
1,Q02,Mann-Whitney U,256.5,0.542825,99,0.404040,0.0,6,0.500000,0.5,0.902823,False
2,Q03,Mann-Whitney U,281.0,0.824342,99,0.767677,1.0,6,0.833333,1.0,0.902823,False
3,Q04,Mann-Whitney U,251.0,0.504679,99,0.444444,0.0,6,0.500000,1.0,0.902823,False
4,Q05,Mann-Whitney U,236.0,0.352164,99,0.161616,0.0,6,0.500000,0.5,0.902823,False
5,Q06,Mann-Whitney U,314.5,0.806269,99,0.656566,1.0,6,0.500000,0.5,0.902823,False
6,Q07,Mann-Whitney U,279.0,0.791281,99,0.232323,0.0,6,0.333333,0.0,0.902823,False
7,Q08,Mann-Whitney U,231.0,0.312881,99,0.282828,0.0,6,0.500000,0.5,0.902823,False
8,Q09,Mann-Whitney U,258.0,0.572841,99,0.686869,1.0,6,0.833333,1.0,0.902823,False
9,Q10,Mann-Whitney U,255.5,0.548384,99,0.747475,1.0,6,1.166667,1.0,0.902823,False



Node: harassment_conflict_friend
Id: Scene4Bedroom.phone.choices
Options:
response
Por la foto.                   1060
Prefiero no hablar de ello.     660
Por nuestra relación.           400
Name: count, dtype: int64
Test: Kruskal-Wallis
Significant: 0/20


...

,id,test,stat,p,n_Enviar otra foto.,mean_diff_Enviar otra foto.,median_diff_Enviar otra foto.,n_Ya no quiero enviar más fotos.,mean_diff_Ya no quiero enviar más fotos.,median_diff_Ya no quiero enviar más fotos.,p_adj,significant
0,Q01,Mann-Whitney U,1142.0,0.259708,42,0.476190,0.0,62,0.741935,1.0,0.746599,False
1,Q02,Mann-Whitney U,1373.0,0.607497,42,0.428571,0.0,62,0.403226,0.0,0.809996,False
2,Q03,Mann-Whitney U,1201.0,0.489972,42,0.642857,1.0,62,0.870968,1.0,0.806488,False
3,Q04,Mann-Whitney U,1282.5,0.893664,42,0.452381,0.0,62,0.435484,0.0,0.955972,False
4,Q05,Mann-Whitney U,1215.0,0.524217,42,0.142857,0.0,62,0.209677,0.0,0.806488,False
5,Q06,Mann-Whitney U,1151.0,0.297761,42,0.523810,0.0,62,0.741935,1.0,0.746599,False
6,Q07,Mann-Whitney U,1423.0,0.382818,42,0.357143,0.0,62,0.161290,0.0,0.765635,False
7,Q08,Mann-Whitney U,1408.5,0.434251,42,0.380952,0.0,62,0.241935,0.0,0.789547,False
8,Q09,Mann-Whitney U,995.5,0.031653,42,0.500000,0.0,62,0.838710,1.0,0.633057,False
9,Q10,Mann-Whitney U,1115.0,0.190669,42,0.595238,0.0,62,0.903226,1.0,0.746599,False



Node: agree_to_meet_harasser
Id: Scene6BedroomRouteA1.phone.choices2
Options:
response
Aceptar.    660
Negarse.    180
Name: count, dtype: int64
Test: Mann-Whitney U
Significant: 0/20


,id,test,stat,p,n_Aceptar.,mean_diff_Aceptar.,median_diff_Aceptar.,n_Negarse.,mean_diff_Negarse.,median_diff_Negarse.,p_adj,significant
0,Q01,Mann-Whitney U,119.0,0.340099,33,0.424242,0.0,9,0.666667,1.0,0.566831,False
1,Q02,Mann-Whitney U,188.0,0.186563,33,0.515152,0.0,9,0.111111,0.0,0.376097,False
2,Q03,Mann-Whitney U,162.0,0.675755,33,0.666667,1.0,9,0.555556,0.0,0.818159,False
3,Q04,Mann-Whitney U,143.5,0.884785,33,0.393939,0.0,9,0.666667,0.0,0.931353,False
4,Q05,Mann-Whitney U,197.0,0.116398,33,0.212121,0.0,9,-0.111111,-1.0,0.376097,False
5,Q06,Mann-Whitney U,215.0,0.034142,33,0.696970,1.0,9,-0.111111,0.0,0.188963,False
6,Q07,Mann-Whitney U,199.5,0.099435,33,0.515152,0.0,9,-0.222222,0.0,0.376097,False
7,Q08,Mann-Whitney U,189.5,0.187961,33,0.454545,0.0,9,0.111111,0.0,0.376097,False
8,Q09,Mann-Whitney U,188.5,0.188048,33,0.606061,0.0,9,0.111111,0.0,0.376097,False
9,Q10,Mann-Whitney U,193.5,0.148376,33,0.757576,1.0,9,0.000000,0.0,0.376097,False



Node: confess_to_parents
Id: Scene6LunchRouteB.interruption.choices
Options:
response
Decir la verdad.    1240
Mentir.              180
Name: count, dtype: int64
Test: Mann-Whitney U
Significant: 0/20


,id,test,stat,p,n_Decir la verdad.,mean_diff_Decir la verdad.,median_diff_Decir la verdad.,n_Mentir.,mean_diff_Mentir.,median_diff_Mentir.,p_adj,significant
0,Q01,Mann-Whitney U,252.5,0.633944,62,0.693548,1.0,9,1.000000,1.0,0.943000,False
1,Q02,Mann-Whitney U,227.5,0.327794,62,0.338710,0.0,9,0.555556,0.0,0.602160,False
2,Q03,Mann-Whitney U,224.0,0.331188,62,0.758065,1.0,9,1.333333,1.0,0.602160,False
3,Q04,Mann-Whitney U,370.0,0.096353,62,0.548387,0.0,9,-0.111111,0.0,0.481765,False
4,Q05,Mann-Whitney U,294.5,0.772101,62,0.177419,0.0,9,0.111111,0.0,1.000000,False
5,Q06,Mann-Whitney U,359.0,0.152123,62,0.693548,1.0,9,0.222222,0.0,0.507077,False
6,Q07,Mann-Whitney U,275.5,0.954118,62,0.096774,0.0,9,0.222222,0.0,1.000000,False
7,Q08,Mann-Whitney U,279.0,1.000000,62,0.225806,0.0,9,0.222222,0.0,1.000000,False
8,Q09,Mann-Whitney U,290.0,0.848184,62,0.725806,1.0,9,0.888889,0.0,1.000000,False
9,Q10,Mann-Whitney U,247.5,0.567936,62,0.741935,1.0,9,1.111111,1.0,0.943000,False


In [256]:
target_nodes = {
    "Scene1Bedroom1.computer2.choices2": "school_year",
}

user_school_year = {}

for trace in traces:
    obj_id = trace.get("object", {}).get("id", "")

    if obj_id == CHOICE_OBJ_ID:
        extensions = trace.get("result", {}).get("extensions", {})

        node = extensions.get(NODE_EXT_ID)
        response = extensions.get(RESPONSE_EXT_ID)
        user = trace.get("actor", {}).get("account", {}).get("name")

        if node in target_nodes and response and user:
            # El primer número de la respuesta es el año
            match = re.search(r"\d+", response)

            if match:
                school_year = int(match.group())

                user_school_year[user] = int(match.group())

print(user_school_year)

{'684837aae48b5a00221a37c9_newj': 2, '684837aae48b5a00221a37c9_fwme': 2, '684837aae48b5a00221a37c9_vboy': 4, '684837aae48b5a00221a37c9_cgph': 3, '684837aae48b5a00221a37c9_ndkj': 3, '684837aae48b5a00221a37c9_qolu': 2, '684837aae48b5a00221a37c9_ntvv': 1, '684837aae48b5a00221a37c9_qijz': 2, '684837aae48b5a00221a37c9_wiuv': 1, '684837aae48b5a00221a37c9_xnjp': 2, '684837aae48b5a00221a37c9_nuwx': 2, '684837aae48b5a00221a37c9_rcee': 2, '684837aae48b5a00221a37c9_oxqv': 2, '684837b0e48b5a00221a37d0_cpzn': 2, '684837b0e48b5a00221a37d0_sffq': 2, '684837b0e48b5a00221a37d0_lnbt': 2, '684837b0e48b5a00221a37d0_wulv': 2, '684837aae48b5a00221a37c9_yjbq': 2, '684837aae48b5a00221a37c9_pkpj': 2, '684837aae48b5a00221a37c9_grwp': 2, '684837aae48b5a00221a37c9_mdsy': 2, '684837aae48b5a00221a37c9_dlez': 2, '684837b0e48b5a00221a37d0_pdpf': 3, '684837b0e48b5a00221a37d0_qlzj': 2, '684837b0e48b5a00221a37d0_hgry': 2, '684837b0e48b5a00221a37d0_lntj': 3, '684837b0e48b5a00221a37d0_ossy': 2, '684837aae48b5a00221a37c9_u

In [257]:
users_df = global_df[["user_id", "school_year"]].drop_duplicates("user_id")

# Comparar school_year del nodo con el de global_df
users_df ["school_year_node"] = users_df ["user_id"].map(user_school_year)

# Solo usuarios para los que tenemos ambos valores
comparison = users_df .dropna(subset=["school_year", "school_year_node"])

# Coincidencias
comparison["match"] = comparison["school_year"] == comparison["school_year_node"]

n_coincidences = comparison["match"].sum()
n_total = len(comparison)

print(f"Coinciden: {n_coincidences}/{n_total}")
print(f"Porcentaje: {n_coincidences / n_total * 100:.2f}%")


Coinciden: 76/104
Porcentaje: 73.08%
